*0.2 Math / ML basics*

# dot product

**The situation.** The search box now has vectors. A query comes in and must be scored against every article. You need one number per article that says *how much it agrees with the query*, and you need it fast enough to run on every keystroke.

**The dot product.** Multiply the two vectors number by number and add everything up. If both vectors point the same way, the products are positive and the sum is large. If they point in different directions, the sum is small or negative. One line of NumPy, and it is what every vector database computes under the hood.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Score four articles against one query.** Embed them, then `articles @ query` computes every dot product at once.

In [2]:
import numpy as np
from openai import OpenAI

client = OpenAI(timeout=30)
articles = [
    "Duplicate payment on your statement",
    "How to request a refund",
    "How to change your profile photo",
    "Two-factor authentication setup",
]
query = "my card was charged twice"
response = client.embeddings.create(model="text-embedding-3-small", input=articles + [query])
vectors = []
for item in response.data:
    vectors.append(item.embedding)
vectors = np.array(vectors, dtype=np.float32)
article_vectors, query_vector = vectors[:-1], vectors[-1]

scores = article_vectors @ query_vector  # four dot products in one operation
order = np.argsort(-scores)  # highest score first
for position in order:
    print(f"{scores[position]:.3f}  {articles[position]}")
assert articles[order[0]] == "Duplicate payment on your statement"

0.561  Duplicate payment on your statement
0.242  Two-factor authentication setup
0.233  How to request a refund
0.127  How to change your profile photo


**Reading the output.** The refund and duplicate-payment articles score highest, the profile photo lowest. Higher dot product = more agreement. This is the ranking the search page shows.

**By hand, to see what `@` did.** The same number, computed the long way for the top article.

In [3]:
by_hand = 0.0
for a, b in zip(article_vectors[order[0]], query_vector):
    by_hand += float(a) * float(b)
print("multiply and add, 1,536 times:", round(by_hand, 4))
print("NumPy:                        ", round(float(scores[order[0]]), 4))
assert abs(by_hand - scores[order[0]]) < 1e-3

multiply and add, 1,536 times: 0.5615
NumPy:                         0.5615


```
query    [ 0.2, -0.1,  0.4 ]
article  [ 0.3, -0.2,  0.5 ]
         ─────────────────
          0.06 + 0.02 + 0.20  =  0.28   ← one number: how much they agree
```

**The rule to remember.** The dot product is the score. Everything else in vector search — indexes, GPUs, approximate search — is about computing millions of dot products quickly.

| Use it when | Don't when | Instead use |
|---|---|---|
| vectors are unit length (OpenAI, most modern embedding models) | vectors have different lengths — long texts win regardless of meaning | cosine similarity (next item) |

**Watch out**
- Check your vector database's metric setting matches the model: `dot` for unit-length vectors, `cosine` otherwise. A mismatch silently ranks wrong.
- Scores are only comparable within one model. 0.55 from one model and 0.55 from another mean different things.
- Never loop in Python over vectors; `@` is a thousand times faster.